# SEBAL Notebook 4: Surface Variables and Energy Balance Inputs

**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook prepares the core Landsat-derived surface variables required for SEBAL processing over the Mississippi Delta. The workflow computes NDVI, surface albedo, land surface temperature (LST), net radiation (Rn), and soil heat flux (G), which serve as foundational inputs for subsequent anchor-pixel selection and flux calibration steps.

## 1. Import Required Python Libraries and Utility Functions

In [1]:
import os
import rasterio
import numpy as np
from Utility import get_file_dataframe
from Utility import load_rasters
from Utility import calculate_NDVI
from Utility import calculate_albedo
from Utility import extract_mask_mean
from Utility import read_raster
from Utility import write_raster
from Utility import check_raster

## 2. Set SEBAL working directory

In [2]:
os.chdir("..")
os.getcwd()

'G:\\Collaborations\\Mentee\\UF_Anitha Madapakula\\Scripts\\Python\\SEBAL_20200119_MSDelta'

## 3. Define bands, directories and input files

In [3]:
clip_dir = "03_clip_landsat"
# output folder
out_dir = "04_indices"
os.makedirs(out_dir, exist_ok=True)
region = 'MSDelta'
proc_dir = "03_processed_met"
hour_str = '16'
bands = ["blue","green","red","nir08","swir16","swir22","lwir"]
landsat_file = "landsat_downloaded_files.csv"

## 4. Load clipped rasters

In [4]:
rasters, srcs, date = load_rasters(landsat_file, clip_dir, bands)

                          scene_id      band  \
0  LE07_L2SP_023037_20200119_02_T1      blue   
1  LE07_L2SP_023037_20200119_02_T1     green   
2  LE07_L2SP_023037_20200119_02_T1       red   
3  LE07_L2SP_023037_20200119_02_T1     nir08   
4  LE07_L2SP_023037_20200119_02_T1    swir16   
5  LE07_L2SP_023037_20200119_02_T1    swir22   
6  LE07_L2SP_023037_20200119_02_T1      lwir   
7  LE07_L2SP_023037_20200119_02_T1  qa_pixel   
8  LE07_L2SP_023036_20200119_02_T1      blue   
9  LE07_L2SP_023036_20200119_02_T1     green   

                                                file  
0  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
1  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
2  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
3  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
4  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
5  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
6  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
7  G:\Collaborations\Mentee\UF_

## 5. Normalized Vedigtation Difference Index (NDVI)
### 5.1 Calculate NDVI
$$
NDVI = \frac{NIR - RED}{NIR + RED}
$$

In [5]:
ndvi = calculate_NDVI(rasters)

NDVI min/max (valid): -0.8984612226486206 0.9352710843086243
NDVI nodata count: 19250102


### 5.2 Save NDVI

In [6]:
# output filename
out_path = f"{out_dir}/NDVI_{date}_{region}.tif"
# use metadata from any band source (red is fine)
meta = srcs["red"].meta.copy()
write_raster(raster_path=out_path, profile=meta, value=ndvi)

Saved: 04_indices/NDVI_20200119_MSDelta.tif


### 5.3 Quick validation

In [7]:
ndvi_check = rasterio.open(out_path).read(1)
print("shape:", ndvi_check.shape)
print("min:", ndvi_check[ndvi_check > -9999].min())
print("max:", ndvi_check[ndvi_check > -9999].max())

shape: (9905, 3618)
min: -0.8984612
max: 0.9352711


## 6. Surface Albedo (α)
### 6.1 Calculate Albedo
$$
\alpha = 0.356 \rho_{BLUE} + 0.130 \rho_{RED}  + 0.373 \rho_ {NIR} + 0.085 \rho_{SWIR1}  + 0.072 \rho_{SWIR2} - 0.0018
$$
where SWIR1 = Shortwave Infrared Band 1, SWIR2 = Shortwave Infrared Band 2

In [8]:
albedo = calculate_albedo(rasters)

Albedo min/max (valid): 7653.2890625 34219.86328125
Albedo nodata count: 19250087


### 6.2 Save Albedo

In [9]:
# output filename
out_path = f"{out_dir}/Albedo_raw_{date}_{region}.tif"
write_raster(raster_path=out_path, profile=meta, value=albedo)

Saved: 04_indices/Albedo_raw_20200119_MSDelta.tif


### 6.3 Quick validation

In [10]:
albedo_check = rasterio.open(out_path).read(1)
print("shape:", albedo_check.shape)
print("min:", albedo_check[albedo_check > -9999].min())
print("max:", albedo_check[albedo_check > -9999].max())

shape: (9905, 3618)
min: 7653.289
max: 34219.863


### 6.4 Final SEBAL Albedo

In [11]:
# SR scaling for Landsat Collection 2 Level-2
scale  = 0.0000275
offset = -0.2

blue_ref  = rasters["blue"].astype("float32")  * scale + offset
red_ref   = rasters["red"].astype("float32")   * scale + offset
nir_ref   = rasters["nir08"].astype("float32") * scale + offset
swir1_ref = rasters["swir16"].astype("float32")* scale + offset
swir2_ref = rasters["swir22"].astype("float32")* scale + offset

valid = (blue_ref > 0) & (red_ref > 0) & (nir_ref > 0) & (swir1_ref > 0) & (swir2_ref > 0)

albedo_ref = np.full(red_ref.shape, -9999.0, dtype=np.float32)
albedo_ref[valid] = (
    0.356*blue_ref[valid] +
    0.130*red_ref[valid] +
    0.373*nir_ref[valid] +
    0.085*swir1_ref[valid] +
    0.072*swir2_ref[valid] -
    0.0018
)

print("Albedo (reflectance) min/max:", albedo_ref[valid].min(), albedo_ref[valid].max())

Albedo (reflectance) min/max: 0.0071317246 0.7360462


In [12]:
out_path_final = f"{out_dir}/ALBEDO_{date}_{region}.tif"
write_raster(raster_path=out_path_final, profile=meta, value=albedo_ref)

Saved: 04_indices/ALBEDO_20200119_MSDelta.tif


In [13]:
albedo_ref[valid] = np.clip(albedo_ref[valid], 0.02, 0.35)

print("Clamped min/max:", 
      albedo_ref[valid].min(), 
      albedo_ref[valid].max())

Clamped min/max: 0.02 0.35


## 7. LST (Land Surface Temperature)
### 7.1 Locate the LST band

In [14]:
# Locate LWIR raster
lwir_tif = os.path.join(clip_dir, f"lwir_{date}_{region}.tif")

# read raster
lst_raw, profile, nod = read_raster(lwir_tif)

# assign fallback nodata
nod = nod if nod is not None else -9999.0

### 7.2 Convert to Celsius
The Landsat Level-2 surface temperature is stored as scaled digital numbers (DN).  
Conversion follows:
$$
T_K = DN \times 0.00341802 + 149.0
$$
$$
T_C = T_K - 273.15
$$

In [15]:
# Convert Landsat Collection-2 LST stored integers to Celsius

scale = 0.00341802
offset = 149.0

valid = (lst_raw != nod)

lst_k = np.full(lst_raw.shape, -9999.0, dtype="float32")
lst_k[valid] = lst_raw[valid] * scale + offset

lst_c = np.full(lst_raw.shape, -9999.0, dtype="float32")
lst_c[valid] = lst_k[valid] - 273.15

print("LST C min/max:",
      lst_c[lst_c > -9999].min(),
      lst_c[lst_c > -9999].max())

LST C min/max: -25.03424 21.652496


### 7.3 Save the  LST

In [16]:
# output filename
out_path = f"{out_dir}/LST_{date}_{region}.tif"
write_raster(raster_path=out_path, profile=meta, value=lst_c)

Saved: 04_indices/LST_20200119_MSDelta.tif


### 7.4 Mask LST using NDVI > 0

In [17]:
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"
ndvi, ndvi_profile, ndvi_nod = read_raster(ndvi_path)

mask = (ndvi > 0.0) & (lst_c > -9999)
lst_c2 = np.full(lst_c.shape, -9999.0, dtype="float32")
lst_c2[mask] = lst_c[mask]

print("Masked LST C min/max:", 
      lst_c2[lst_c2 > -9999].min(), 
      lst_c2[lst_c2 > -9999].max())

Masked LST C min/max: -23.49614 21.652496


### 7.5 Save the masked LST

In [18]:
out_path2 = f"{out_dir}/LST_C_{date}_{region}.tif"
write_raster(raster_path=out_path2, profile=meta, value=lst_c2)

Saved: 04_indices/LST_C_20200119_MSDelta.tif


### 7.6 Check

In [19]:
print("Valid LST before mask:", np.sum(lst_c > -9999))
print("Valid LST after mask: ", np.sum(lst_c2 > -9999))
print("Removed pixels:       ", np.sum((lst_c > -9999) & (lst_c2 == -9999)))

Valid LST before mask: 16586203
Valid LST after mask:  15422292
Removed pixels:        1163911


## 8. Net Radiation (SEBAL)

Net radiation is computed as:

$$
R_n = R_s^\downarrow (1 - \alpha) + R_L^\downarrow - R_L^\uparrow
$$

where:

• $R_s^\downarrow$ = incoming shortwave radiation (AORC)  
• $\alpha$ = surface albedo  
• $R_L^\downarrow$ = incoming longwave radiation  
• $R_L^\uparrow$ = outgoing longwave radiation
### 8.1 Load inputs

In [20]:
# ---- file paths
rs_path   = f"{proc_dir}/Rs_{date}_{hour_str}Z_{region}.tif"
ta_path   = f"{proc_dir}/TA_{date}_{hour_str}Z_{region}.tif"
alb_path  = f"{out_dir}/ALBEDO_{date}_{region}.tif"
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"    # <- masked LST preferred
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"

# ---- read rasters
# read Rs raster
Rs, rs_profile, rs_nodata = read_raster(rs_path)

# read TA raster
TA, ta_profile, ta_nodata = read_raster(ta_path)

# read albedo raster
alb, alb_profile, alb_nodata = read_raster(alb_path)

# read LST raster
lstC, lst_profile, lst_nodata = read_raster(lst_path)

# read NDVI raster
ndvi, ndvi_profile, ndvi_nodata = read_raster(ndvi_path)

# ---- valid mask
valid = (Rs > -9999) & (TA > -9999) & (alb > -9999) & (lstC > -9999)

print("Valid pixels:", int(valid.sum()))

Valid pixels: 15410412


### 8.2 Compute Rn
- Compute net radiation (Rn) using a simplified SEBAL formulation.
- Surface albedo is taken from broadband albedo, surface temperature is taken from LST,
- and surface emissivity is assigned from NDVI-based classes.
- Incoming longwave radiation is approximated using the Stefan-Boltzmann law with air temperature,
- while outgoing longwave radiation is computed from surface emissivity and surface temperature.
- This is a simplified implementation intended for stable first-pass SEBAL calculations.

In [21]:
sigma = 5.67e-8

# create empty Kelvin air-temperature array (same size as TA)
TaK  = np.full(TA.shape, np.nan, dtype="float32")
 # create empty Kelvin surface-temperature array (same size as LST)
LstK = np.full(lstC.shape, np.nan, dtype="float32")

TaK[valid]  = TA[valid]              # copy valid TA pixels directly (already in Kelvin)
LstK[valid] = lstC[valid] + 273.15   # convert valid LST pixels from Celsius → Kelvin

RL_down = np.full(Rs.shape, np.nan, dtype="float32") # allocate array for incoming longwave radiation
RL_up   = np.full(Rs.shape, np.nan, dtype="float32") # allocate array for outgoing longwave radiation

# --- emissivity (NDVI-based, stable SEBAL version)
emiss = np.full(ndvi.shape, np.nan, dtype="float32")

# SEBAL simple scheme
# assign high emissivity for dense vegetation pixels
emiss[(ndvi > 0.5)] = 0.99
# assign moderate emissivity for mixed vegetation pixels
emiss[(ndvi > 0.2) & (ndvi <= 0.5)] = 0.97
# assign lower emissivity for bare soil / sparse vegetation
emiss[(ndvi <= 0.2)] = 0.95

# compute incoming longwave radiation using Stefan–Boltzmann law
RL_down[valid] = sigma * (TaK[valid] ** 4)
# compute outgoing longwave using emissivity + surface temperature
RL_up[valid]   = emiss[valid] * sigma * (LstK[valid] ** 4)


 # initialize net radiation array with nodata
Rn = np.full(Rs.shape, -9999.0, dtype="float32")
 # compute net radiation (SEBAL equation)
Rn[valid] = Rs[valid] * (1.0 - alb[valid]) + RL_down[valid] - RL_up[valid]

# quick range check for sanity
print("Rn valid:", np.sum(Rn > -9999))
print("FIXED Rn min/max:", float(Rn[Rn>-9999].min()), float(Rn[Rn>-9999].max()))

Rn valid: 15410412
FIXED Rn min/max: 147.48837280273438 415.78106689453125


### 8.3 Save Rn

In [22]:

# define output filename for net radiation raster
out_rn = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"
write_raster(raster_path=out_rn, profile=rs_profile, value=Rn)

Saved: 04_indices/Rn_20200119_16Z_MSDelta.tif


##  9 Soil Heat Flux (G)

SEBAL uses an empirical relationship. The most common form (Bastiaanssen-type) is:

$$
𝐺 = 𝑅_𝑛 \frac{𝑇_𝑠}{\alpha} (0.0038𝛼 + 0.0074𝛼^2) (1 − 0.98𝑁𝐷𝑉𝐼^4)
$$

### 9.1 Load inputs (Rn, LST, Albedo, NDVI)

In [23]:
# define path to net radiation raster
rn_path   = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"

# define path to surface temperature raster (Celsius)
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"

# define path to surface albedo raster
alb_path  = f"{out_dir}/ALBEDO_{date}_{region}.tif"

# define path to NDVI raster
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"


## load net radiation raster
Rn, profile, rn_nodata = read_raster(rn_path)

# load surface temperature raster (°C)
LstC, _, lst_nodata = read_raster(lst_path)

# load albedo raster
Alb, _, alb_nodata = read_raster(alb_path)

# load NDVI raster
Ndvi, _, ndvi_nodata = read_raster(ndvi_path)

# create physically valid NDVI copy for anchor-pixel selection
Ndvi2 = np.where((Ndvi > -1.0) & (Ndvi < 1.0), Ndvi, np.nan)

# build mask of valid pixels across all inputs
valid = (Rn > -9999) & (LstC > -9999) & (Alb > -9999) & (Ndvi > -9999)

# print count for quick verification
print("Valid pixels:", int(valid.sum()))

Valid pixels: 15410412


### 9.2 Compute G

In [24]:
# initialize G raster
G = np.full(Rn.shape, -9999.0, dtype="float32")

# protect against very small albedo values
alb_safe = np.where((alb > 0.01) & valid, alb, np.nan)

# use cleaned NDVI
ndvi_safe = np.where(valid, Ndvi2, np.nan)

# compute SEBAL soil heat flux using LST in Celsius
G_calc = Rn * (
    LstC *
    (0.0038 + 0.0074 * alb_safe) *
    (1 - 0.98 * ndvi_safe**4)
)

# assign valid pixels
G[valid] = G_calc[valid]

# clean valid mask
g_valid = valid & np.isfinite(G) & (G > -9999) & np.isfinite(Rn) & (Rn > 0)

# diagnostics
ratio = np.full(Rn.shape, np.nan, dtype="float32")
ratio[g_valid] = G[g_valid] / Rn[g_valid]

### 9.3 Sanity check

In [25]:
# print clean diagnostics
print("Raw G valid:", np.sum(g_valid))
print("Raw G min/max:", float(G[g_valid].min()), float(G[g_valid].max()))
print("Raw G/Rn min/max:", float(np.nanmin(ratio)), float(np.nanmax(ratio)))

Raw G valid: 15410408
Raw G min/max: -56.29561996459961 27.165998458862305
Raw G/Rn min/max: -0.135397270321846 0.10463870316743851


### 9.4 Clamp G (recommended)

In [26]:
# Apply a practical physical constraint to G/Rn:
# enforce nonnegative soil heat flux and cap unrealistically high ratios
# before saving the final soil heat flux raster.
ratio_clamped = np.clip(ratio, 0.0, 0.35)

# build final soil heat flux raster
G2 = np.full(Rn.shape, -9999.0, dtype="float32")
G2[g_valid] = Rn[g_valid] * ratio_clamped[g_valid]

# final diagnostics
print("G2 valid:", np.sum(G2 > -9999))
print("G2 min/max:", float(G2[G2 > -9999].min()), float(G2[G2 > -9999].max()))
print("G2/Rn min/max:",
      float(np.nanmin(ratio_clamped)),
      float(np.nanmax(ratio_clamped)))

G2 valid: 15410408
G2 min/max: 0.0 27.165998458862305
G2/Rn min/max: 0.0 0.10463870316743851


### 9.5 Save G

In [27]:
# define output filename for soil heat flux raster
out_g = f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif"
G = G2
write_raster(raster_path=out_g, profile=profile, value=G)

Saved: 04_indices/G_20200119_16Z_MSDelta.tif
